In [1]:
import torch
print(torch.cuda.is_available())

True


## CNN Image Classification (Overfitting + CUDA OOM)

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ✅ Device setup (works for both CPU & GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ✅ Transform
transform = transforms.Compose([transforms.ToTensor()])

# ✅ Fake dataset
train_data = datasets.FakeData(size=1400, image_size=(3, 32, 32), transform=transform)
val_data = datasets.FakeData(size=300, image_size=(3, 32, 32), transform=transform)

# ✅ Loaders
train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
train_loader2 = DataLoader(train_data, batch_size=8, shuffle=True)  # fixed batch
val_loader = DataLoader(val_data, batch_size=16, shuffle=False)

# ✅ CNN Model (FIXED)
class BigCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 5)
        self.conv2 = nn.Conv2d(64, 128, 5)
        
        # ✅ FIXED SIZE (128 * 5 * 5 = 3200)
        self.fc1 = nn.Linear(128*5*5, 512)
        self.fc2 = nn.Linear(512, 10)
    
    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.max_pool2d(x, 2)
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# ==============================
# 🔴 BEFORE FIX (Overfitting)
# ==============================

model = BigCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

print("\n=== BEFORE FIX: Overfitting ===")

for epoch in range(2):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        
        out = model(data)
        loss = criterion(out, target)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if batch_idx % 5 == 0:
            print(f'Epoch {epoch}, Loss: {loss.item():.2f}')
    
    # Validation
    model.eval()
    val_loss, correct = 0, 0
    
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            
            out = model(data)
            val_loss += criterion(out, target).item()
            pred = out.argmax(1)
            correct += pred.eq(target).sum().item()
    
    print(f'Val Loss: {val_loss/len(val_loader):.2f}, Val Acc: {correct/len(val_data)*100:.1f}%')

# ==============================
# 🟢 AFTER FIX (Improved)
# ==============================

# ✅ New model with dropout
model2 = BigCNN()

model2.fc1 = nn.Sequential(
    nn.Linear(128*5*5, 512),
    nn.ReLU(),
    nn.Dropout(0.5)
)

model2 = model2.to(device)

optimizer2 = optim.Adam(model2.parameters(), lr=0.001, weight_decay=0.001)

print("\n=== AFTER FIX: Smaller batch + Dropout ===")

for epoch in range(3):
    model2.train()
    for data, target in train_loader2:
        data, target = data.to(device), target.to(device)
        
        out = model2(data)
        loss = criterion(out, target)
        
        optimizer2.zero_grad()
        loss.backward()
        optimizer2.step()
    
    # Validation
    model2.eval()
    correct = 0
    
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            
            out = model2(data)
            pred = out.argmax(1)
            correct += pred.eq(target).sum().item()
    
    print(f'Epoch {epoch}, Val Acc: {correct/len(val_data)*100:.1f}%')

Using device: cuda

=== BEFORE FIX: Overfitting ===
Epoch 0, Loss: 2.32
Epoch 0, Loss: 2.30
Epoch 0, Loss: 2.31
Epoch 0, Loss: 2.31
Epoch 0, Loss: 2.30
Epoch 0, Loss: 2.30
Epoch 0, Loss: 2.30
Epoch 0, Loss: 2.29
Epoch 0, Loss: 2.31
Epoch 0, Loss: 2.31
Epoch 0, Loss: 2.30
Epoch 0, Loss: 2.34
Epoch 0, Loss: 2.31
Epoch 0, Loss: 2.30
Epoch 0, Loss: 2.31
Epoch 0, Loss: 2.29
Epoch 0, Loss: 2.30
Epoch 0, Loss: 2.31
Val Loss: 2.30, Val Acc: 11.0%
Epoch 1, Loss: 2.32
Epoch 1, Loss: 2.30
Epoch 1, Loss: 2.29
Epoch 1, Loss: 2.31
Epoch 1, Loss: 2.31
Epoch 1, Loss: 2.30
Epoch 1, Loss: 2.31
Epoch 1, Loss: 2.32
Epoch 1, Loss: 2.29
Epoch 1, Loss: 2.29
Epoch 1, Loss: 2.29
Epoch 1, Loss: 2.28
Epoch 1, Loss: 2.29
Epoch 1, Loss: 2.31
Epoch 1, Loss: 2.32
Epoch 1, Loss: 2.29
Epoch 1, Loss: 2.29
Epoch 1, Loss: 2.30
Val Loss: 2.30, Val Acc: 11.0%

=== AFTER FIX: Smaller batch + Dropout ===
Epoch 0, Val Acc: 11.0%
Epoch 1, Val Acc: 11.0%
Epoch 2, Val Acc: 11.0%


In [6]:
!pip install pandas numpy scikit-learn

## LSTM Sentiment Analysis (NaN/missing data)

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

# =========================
# ✅ DEVICE SETUP
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================
# 📊 CREATE DATASET
# =========================
np.random.seed(42)

data = {
    'text': ['good', 'bad', np.nan, 'ok', np.nan, 'great'] * 833,
    'label': [1, 0, np.nan, 0, np.nan, 1] * 833
}

df = pd.DataFrame(data)

print("\nDataset Preview:")
print(df.head())
print(f"NaNs -> Text: {df['text'].isna().sum()}, Labels: {df['label'].isna().sum()}")

# =========================
# 📦 DATASET CLASS
# =========================
class TextDataset(Dataset):
    def __init__(self, df):
        self.texts = df['text'].fillna('unknown').astype(str).values
        self.labels = df['label'].fillna(0).astype(int).values

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        length = len(self.texts[idx])
        return (
            torch.tensor([length], dtype=torch.float32),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

# =========================
# 🤖 MODEL
# =========================
class SentimentLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=32, batch_first=True)
        self.fc = nn.Linear(32, 2)

    def forward(self, x):
        x = x.unsqueeze(1)        # (batch, 1, 1)
        x, _ = self.lstm(x)       # (batch, 1, 32)
        x = x[:, -1, :]           # (batch, 32)
        return self.fc(x)

# =========================
# 🔴 BEFORE CLEANING
# =========================
dataset = TextDataset(df)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

model = SentimentLSTM().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

print("\n=== BEFORE CLEANING ===")

for epoch in range(2):
    model.train()
    total_loss = 0

    for data, target in loader:
        data, target = data.to(device), target.to(device)

        outputs = model(data)
        loss = criterion(outputs, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")

# =========================
# 🟢 AFTER CLEANING
# =========================
print("\n=== AFTER CLEANING ===")

df_clean = df.copy()

# Feature engineering
df_clean['text_len'] = df_clean['text'].fillna('unknown').astype(str).str.len()

# Remove outliers (IQR)
q1, q3 = df_clean['text_len'].quantile([0.25, 0.75])
iqr = q3 - q1

df_clean = df_clean[
    (df_clean['text_len'] >= q1 - 1.5 * iqr) &
    (df_clean['text_len'] <= q3 + 1.5 * iqr)
]

# Fill missing labels
df_clean['label'] = df_clean['label'].fillna(df_clean['label'].median()).astype(int)

# New dataset
dataset_clean = TextDataset(df_clean)
loader_clean = DataLoader(dataset_clean, batch_size=32, shuffle=True)

model2 = SentimentLSTM().to(device)
optimizer2 = optim.Adam(model2.parameters(), lr=0.001)

# Training after cleaning
for epoch in range(3):
    model2.train()
    total_loss, correct = 0, 0

    for data, target in loader_clean:
        data, target = data.to(device), target.to(device)

        outputs = model2(data)
        loss = criterion(outputs, target)

        optimizer2.zero_grad()
        loss.backward()
        optimizer2.step()

        total_loss += loss.item()
        preds = outputs.argmax(1)
        correct += preds.eq(target).sum().item()

    acc = correct / len(dataset_clean)

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader_clean):.4f}, Accuracy: {acc:.2%}")

Using device: cuda

Dataset Preview:
   text  label
0  good    1.0
1   bad    0.0
2   NaN    NaN
3    ok    0.0
4   NaN    NaN
NaNs -> Text: 1666, Labels: 1666

=== BEFORE CLEANING ===
Epoch 1, Loss: 0.4591
Epoch 2, Loss: 0.0368

=== AFTER CLEANING ===
Epoch 1, Loss: 0.6380, Accuracy: 66.67%
Epoch 2, Loss: 0.6235, Accuracy: 66.67%
Epoch 3, Loss: 0.5851, Accuracy: 66.67%


## Feedforward Net (Slow training)

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
import time

# Fake regression dataset (~2000 samples)
X = torch.randn(2000, 10)
y = X.sum(dim=1, keepdim=True) + 0.1 * torch.randn(2000, 1)

train_loader = torch.utils.data.DataLoader(list(zip(X[:1400], y[:1400])), batch_size=1, shuffle=True)  # Tiny batches = slow
val_loader = torch.utils.data.DataLoader(list(zip(X[1400:1700], y[1400:1700])), batch_size=1)

print("=== BEFORE OPTIMIZATION: Slow CPU training ===")
class SlowNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 1)
    
    def forward(self, x):
        return self.fc3(torch.relu(self.fc2(torch.relu(self.fc1(x)))))

model_slow = SlowNet()  # CPU only
optimizer_slow = optim.SGD(model_slow.parameters(), lr=0.01)  # SGD slow convergence
criterion = nn.MSELoss()

start = time.time()
for epoch in range(5):
    model_slow.train()
    epoch_start = time.time()
    for data, target in train_loader:
        out = model_slow(data)
        loss = criterion(out, target)
        optimizer_slow.zero_grad()
        loss.backward()
        optimizer_slow.step()
    epoch_time = time.time() - epoch_start
    print(f'Epoch {epoch}, Time: {epoch_time:.1f}s, Loss: {loss.item():.3f}')
print(f"Total time: {time.time()-start:.1f}s")

print("\n=== AFTER OPTIMIZATION: Batches + Adam ===")
# Fix 1: Bigger batches
train_loader_fast = torch.utils.data.DataLoader(list(zip(X[:1400], y[:1400])), 
                                               batch_size=32, shuffle=True)

model_fast = SlowNet()
optimizer_fast = optim.Adam(model_fast.parameters(), lr=0.001)  # Adam faster

start = time.time()
for epoch in range(5):
    model_fast.train()
    epoch_start = time.time()
    for batch_idx, (data, target) in enumerate(train_loader_fast):
        out = model_fast(data)
        loss = criterion(out, target)
        optimizer_fast.zero_grad()
        loss.backward()
        optimizer_fast.step()
    epoch_time = time.time() - epoch_start
    print(f'Epoch {epoch}, Time: {epoch_time:.1f}s, Loss: {loss.item():.3f}')
print(f"Total time: {time.time()-start:.1f}s")


=== BEFORE OPTIMIZATION: Slow CPU training ===
Epoch 0, Time: 2.0s, Loss: 0.043
Epoch 1, Time: 2.2s, Loss: 0.011
Epoch 2, Time: 2.4s, Loss: 0.001
Epoch 3, Time: 1.8s, Loss: 0.000
Epoch 4, Time: 3.1s, Loss: 0.004
Total time: 11.5s

=== AFTER OPTIMIZATION: Batches + Adam ===
Epoch 0, Time: 0.2s, Loss: 3.048
Epoch 1, Time: 0.3s, Loss: 0.181
Epoch 2, Time: 0.2s, Loss: 0.055
Epoch 3, Time: 0.1s, Loss: 0.159
Epoch 4, Time: 0.1s, Loss: 0.060
Total time: 0.9s
